In [23]:
# ===== Compare runs collected with different NUM_PARTITIONS and pick the best =====
# Paste this in ONE Jupyter/IPython cell and run.
# Edit the RUNS list at the bottom to point to your parquet & block_maxima.csv for each run.
# MAX MESSAGE = 5000

import os, glob, json, math
from pathlib import Path
from typing import Dict, List, Optional
import numpy as np
import pandas as pd


In [24]:
# ---------- Helpers to read config / manifests ----------

def _read_any_yaml_json(path: Path) -> dict:
    """Best-effort read of a manifest/config (json or yaml)."""
    if not path.exists():
        return {}
    try:
        if path.suffix.lower() == ".json":
            return json.loads(path.read_text())
        import yaml  # requires pyyaml
        return yaml.safe_load(path.read_text()) or {}
    except Exception:
        return {}

def _find_key_ci(d: dict, key_substr: str):
    """Case-insensitive lookup by substring anywhere in keys (1st level)."""
    ks = key_substr.lower()
    for k, v in d.items():
        if ks in str(k).lower():
            return v
    return None

def _find_recursively(obj, key_substr: str):
    """Case-insensitive recursive lookup over nested dict/list."""
    ks = key_substr.lower()
    if isinstance(obj, dict):
        for k, v in obj.items():
            if ks in str(k).lower():
                return v
            got = _find_recursively(v, key_substr)
            if got is not None:
                return got
    elif isinstance(obj, list):
        for it in obj:
            got = _find_recursively(it, key_substr)
            if got is not None:
                return got
    return None

def extract_run_config(metrics_dir: Path) -> dict:
    """
    Try to pull interesting knobs (e.g., NUM_PARTITIONS, MAX_MESSAGES, TARGET_RATE)
    from files saved next to block_maxima.csv (pipeline-configmap_*, run_manifest.*).
    """
    cfg: dict = {}

    # pipeline-configmap snapshot (preferred)
    for yml in sorted(metrics_dir.glob("pipeline-configmap*.yaml")):
        d = _read_any_yaml_json(yml)
        if not d: 
            continue
        data = d.get("data", {}) or {}
        cfg.update({
            "NUM_PARTITIONS": data.get("NUM_PARTITIONS"),
            "MAX_MESSAGES": data.get("MAX_MESSAGES"),
            "TARGET_RATE": data.get("TARGET_RATE")
        })

    # run manifest (json / yaml)
    for name in ("run_manifest.json","run_manifest.yaml","run_manifest.yml"):
        d = _read_any_yaml_json(metrics_dir / name)
        if not d:
            continue
        for key in ("NUM_PARTITIONS","MAX_MESSAGES","TARGET_RATE"):
            if not cfg.get(key):
                v = _find_recursively(d, key)
                if v is None:
                    v = _find_recursively(d, key.lower())
                if v is not None:
                    cfg[key] = v

    # Coerce to ints where possible
    for k in ("NUM_PARTITIONS","MAX_MESSAGES","TARGET_RATE"):
        v = cfg.get(k)
        if isinstance(v, str) and v.isdigit():
            cfg[k] = int(v)
        else:
            try:
                cfg[k] = int(v)
            except Exception:
                pass
    return cfg

# ---------- Data loading & feature engineering ----------

def load_parquet_glob(parquet_glob_or_dir: str,
                      needed_cols: Optional[List[str]] = None) -> pd.DataFrame:
    """
    Load multiple parquet files. Accepts a glob "*.parquet" or a directory.
    """
    p = Path(parquet_glob_or_dir)
    files = sorted(str(x) for x in (p.glob("*.parquet") if p.is_dir() else glob.glob(parquet_glob_or_dir)))
    if not files:
        raise FileNotFoundError(f"No parquet files match: {parquet_glob_or_dir}")

    cols = set(needed_cols or [])
    dfs = []
    for f in files:
        try:
            if cols:
                # read schema, pick intersection if possible
                import pyarrow.parquet as pq
                have = set(pq.read_schema(f).names)
                use = list(cols & have)
                if not use:
                    continue
                df = pd.read_parquet(f, columns=use)
            else:
                df = pd.read_parquet(f)
            if not df.empty:
                dfs.append(df)
        except Exception as e:
            print(f"[WARN] failed to read {os.path.basename(f)}: {e}")
    if not dfs:
        raise RuntimeError("No parquet files could be read.")
    return pd.concat(dfs, ignore_index=True)

def _coerce_numeric(df, cols: List[str]):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

def compute_latencies(df: pd.DataFrame) -> Dict[str, pd.Series]:
    """
    Build the three latency series (seconds) from consumer parquet batches.
    """
    out = {}
    _coerce_numeric(df, [
        "producer_timestamp","consumer_receive_timestamp",
        "application_timestamp","application_latency_seconds",
        "end_to_end_latency_seconds"
    ])

    # producer_consumer
    if {"producer_timestamp","consumer_receive_timestamp"}.issubset(df.columns):
        pc = (df["consumer_receive_timestamp"] - df["producer_timestamp"]).dropna()
        out["producer_consumer"] = pc

    # consumer_application
    if "application_latency_seconds" in df.columns:
        ca = pd.to_numeric(df["application_latency_seconds"], errors="coerce").dropna()
        out["consumer_application"] = ca
    elif {"application_timestamp","consumer_receive_timestamp"}.issubset(df.columns):
        ca = (df["application_timestamp"] - df["consumer_receive_timestamp"]).dropna()
        out["consumer_application"] = ca

    # producer_application (end-to-end)
    if "end_to_end_latency_seconds" in df.columns:
        e2e = pd.to_numeric(df["end_to_end_latency_seconds"], errors="coerce").dropna()
        out["producer_application"] = e2e
    elif {"application_timestamp","producer_timestamp"}.issubset(df.columns):
        e2e = (df["application_timestamp"] - df["producer_timestamp"]).dropna()
        out["producer_application"] = e2e

    return out

def clip_clean(v: pd.Series, cap_max_s: Optional[float]) -> pd.Series:
    v = v.replace([np.inf, -np.inf], np.nan).dropna()
    v = v[v >= 0]
    if cap_max_s is not None:
        v = v[v <= cap_max_s]
    return v

def frac_over_L(v: pd.Series, L: float) -> float:
    if len(v) == 0: return np.nan
    return float((v > L).mean())

def q_dict(v: pd.Series, qs=(0.50,0.95,0.99)) -> Dict[str, float]:
    if len(v) == 0:
        return {f"p{int(q*100)}": np.nan for q in qs}
    arr = np.quantile(v.to_numpy(), qs)
    return {f"p{int(q*100)}": float(val) for q, val in zip(qs, arr)}

def basic_health_grouped(df: pd.DataFrame) -> Dict[str, float]:
    """
    Compute duplicate & out-of-order rates correctly:
    - group by (topic, partition), sort by offset, compare within the group
    Also estimate throughput (MB/s) from size_bytes sum over time span.
    """
    h = {"out_of_order_rate": np.nan, "duplicate_rate": np.nan,
         "throughput_MBps": np.nan, "duration_s": np.nan, "n_msgs": len(df)}

    needed = {"topic","partition","offset"}
    if needed.issubset(df.columns):
        rates_ooo, rates_dup = [], []
        for (_, _), g in df.groupby(["topic","partition"], dropna=False):
            gg = g.sort_values("offset")
            off = pd.to_numeric(gg["offset"], errors="coerce")
            dup = off.duplicated(keep="first")
            rates_dup.append(float(dup.mean()))
            if len(off) > 1:
                ooo = (off.values[1:] < off.values[:-1])
                rates_ooo.append(float(ooo.mean()))
            else:
                rates_ooo.append(0.0)
        if rates_ooo:
            h["out_of_order_rate"] = float(np.nanmean(rates_ooo))
        if rates_dup:
            h["duplicate_rate"] = float(np.nanmean(rates_dup))

    # throughput from size_bytes + time span (best-effort)
    tcols = [c for c in ("producer_timestamp","consumer_receive_timestamp") if c in df.columns]
    if tcols:
        t = pd.to_numeric(df[tcols[0]], errors="coerce").dropna()
        if len(t) >= 2:
            dur = float(t.max() - t.min())
            h["duration_s"] = dur if dur > 0 else np.nan

            # Prefer actual bytes if present
            if "size_bytes" in df.columns:
                sz = pd.to_numeric(df["size_bytes"], errors="coerce").dropna()
                bytes_total = float(sz.sum()) if len(sz) else np.nan
                if bytes_total and dur and dur > 0:
                    h["throughput_MBps"] = bytes_total / (1024*1024) / dur
            # Fallback: estimate via avg size
            if (not isinstance(h["throughput_MBps"], float)) or math.isnan(h["throughput_MBps"]):
                if "size_bytes" in df.columns:
                    sz_all = pd.to_numeric(df["size_bytes"], errors="coerce")
                    if sz_all.notna().any():
                        avg_sz = float(sz_all.mean())
                        bytes_total = avg_sz * len(df)
                        h["throughput_MBps"] = bytes_total / (1024*1024) / dur
    return h

# ---------- Run analysis ----------

def analyze_run(parquet_glob_or_dir: str,
                blockmax_csv: str,
                label: str,
                # Tail thresholds (seconds):
                Lpc: float = 0.25,    # producer→consumer
                Le2e: float = 0.50,   # end-to-end
                Lca: float = 0.05,    # consumer→application
                # Allowed tail fraction (SLO): max fraction over L
                slo_pc: float = 0.01, slo_e2e: float = 0.01, slo_ca: float = 0.01,
                cap_max_s: Optional[float] = 5.0,
                warmup_sec: float = 60.0) -> dict: 
    """
    Analyze one run (parquet batches + metrics dir next to block_maxima) and return a rich summary dict.
    """
    need = [
        "index","producer_timestamp","consumer_receive_timestamp",
        "application_timestamp","application_latency_seconds","end_to_end_latency_seconds",
        "size_bytes","target_rate","topic","partition","offset",
        "producer_pod","consumer_pod"
    ]
    df = load_parquet_glob(parquet_glob_or_dir, needed_cols=need)
    for c in ("producer_timestamp","consumer_receive_timestamp"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if {"producer_timestamp","consumer_receive_timestamp"}.issubset(df.columns):
        # non-negative network latency
        df = df[df["consumer_receive_timestamp"] - df["producer_timestamp"] >= 0]

        # drop first warmup_sec of the run (start-up effects)
        t0 = df["producer_timestamp"].min()
        if pd.notna(t0) and warmup_sec and warmup_sec > 0:
            df = df[(df["producer_timestamp"] - t0) >= warmup_sec]

        # optional hard cap for synthetic spikes
        if cap_max_s is not None:
            net = df["consumer_receive_timestamp"] - df["producer_timestamp"]
            df = df[net <= cap_max_s]

    # Latencies
    lats = compute_latencies(df)
    pc = clip_clean(lats.get("producer_consumer", pd.Series([], dtype=float)), cap_max_s)
    ca = clip_clean(lats.get("consumer_application", pd.Series([], dtype=float)), cap_max_s)
    e2e= clip_clean(lats.get("producer_application", pd.Series([], dtype=float)), cap_max_s)

    # Tail fractions
    f_pc  = frac_over_L(pc, Lpc)
    f_ca  = frac_over_L(ca, Lca)
    f_e2e = frac_over_L(e2e, Le2e)

    # Percentiles
    q_pc  = q_dict(pc)
    q_ca  = q_dict(ca)
    q_e2e = q_dict(e2e)

    # Health
    h = basic_health_grouped(df)

    # Config from metrics folder
    metrics_dir = Path(blockmax_csv).parent
    cfg = extract_run_config(metrics_dir)
    num_partitions = cfg.get("NUM_PARTITIONS")
    max_messages   = cfg.get("MAX_MESSAGES")
    target_rate    = cfg.get("TARGET_RATE")

    summary = {
        "label": label,
        "num_partitions": num_partitions,
        "max_messages": max_messages,
        "target_rate": target_rate,
        "paths": {
            "parquet": parquet_glob_or_dir,
            "blockmax": blockmax_csv,
            "metrics_dir": str(metrics_dir)
        },
        "health": h,
        "producer_consumer": {
            "n": int(pc.size), "L": Lpc, "frac_over_L": f_pc, **q_pc,
            "slo": slo_pc, "pass": (f_pc <= slo_pc) if not math.isnan(f_pc) else False
        },
        "consumer_application": {
            "n": int(ca.size), "L": Lca, "frac_over_L": f_ca, **q_ca,
            "slo": slo_ca, "pass": (f_ca <= slo_ca) if not math.isnan(f_ca) else False
        },
        "producer_application": {
            "n": int(e2e.size), "L": Le2e, "frac_over_L": f_e2e, **q_e2e,
            "slo": slo_e2e, "pass": (f_e2e <= slo_e2e) if not math.isnan(f_e2e) else False
        },
    }
    # scoring: prioritize SLO passes, then lowest e2e p99, then higher throughput
    passes = int(summary["producer_consumer"]["pass"]) + int(summary["consumer_application"]["pass"]) + int(summary["producer_application"]["pass"])
    p99_e2e = summary["producer_application"]["p99"]
    thr = summary["health"]["throughput_MBps"]
    summary["_score_tuple"] = (
        passes,
        -p99_e2e if p99_e2e==p99_e2e else float("-inf"),
        thr if thr==thr else float("-inf"),
    )
    summary["_passes"] = passes
    return summary

def flatten(summary: dict) -> dict:
    """Flatten one run summary for tabulation."""
    return {
        "label": summary["label"],
        "num_partitions": summary["num_partitions"],
        "max_messages": summary["max_messages"],
        "target_rate": summary["target_rate"],
        "pc_p95": summary["producer_consumer"]["p95"],
        "pc_p99": summary["producer_consumer"]["p99"],
        "pc_frac_over_L": summary["producer_consumer"]["frac_over_L"],
        "pc_L": summary["producer_consumer"]["L"],
        "pc_pass": summary["producer_consumer"]["pass"],
        "ca_p95": summary["consumer_application"]["p95"],
        "ca_p99": summary["consumer_application"]["p99"],
        "ca_frac_over_L": summary["consumer_application"]["frac_over_L"],
        "ca_L": summary["consumer_application"]["L"],
        "ca_pass": summary["consumer_application"]["pass"],
        "e2e_p95": summary["producer_application"]["p95"],
        "e2e_p99": summary["producer_application"]["p99"],
        "e2e_frac_over_L": summary["producer_application"]["frac_over_L"],
        "e2e_L": summary["producer_application"]["L"],
        "e2e_pass": summary["producer_application"]["pass"],
        "throughput_MBps": summary["health"]["throughput_MBps"],
        "duplicate_rate": summary["health"]["duplicate_rate"],
        "out_of_order_rate": summary["health"]["out_of_order_rate"],
        "n_msgs": summary["health"]["n_msgs"],
        "passes_total": summary["_passes"],
        "parquet": summary["paths"]["parquet"],
        "metrics_dir": summary["paths"]["metrics_dir"],
    }


In [25]:
def choose_best_by_latency_and_slo(summaries: List[dict]) -> dict:
    """Pick the best run by (passes desc, e2e p99 asc, throughput desc)."""
    if not summaries:
        raise ValueError("No run summaries")
    return max(summaries, key=lambda s: s["_score_tuple"])

# ---------- Configure your runs here ----------
# Provide one dict per run (each run corresponds to a given NUM_PARTITIONS).
# 'parquet' should point to the parquet glob or directory for that run
# 'blockmax' should point to the matching block_maxima.csv inside the run’s metrics folder

RUNS = [
    # EXAMPLES — replace with your paths:
     dict(
         label="partitions_36",
         parquet="../20250910_143637/consumer/consumer-sts-0_consumer-result/processed/2025-09-10/batch_consumer-sts-*.parquet",
         blockmax="../20250910_143637/merge/merge-sts-0_merge-metrics/2025-09-10_00-37-32/block_maxima.csv",
     ),
    dict(
         label="partitions_72",
         parquet="../20250911_125217/consumer/consumer-sts-0_consumer-result/processed/2025-09-10/batch_consumer-sts-*.parquet",
         blockmax="../20250911_125217/merge/merge-sts-0_merge-metrics/2025-09-10_22-25-23/block_maxima.csv",
     ),
     dict(
         label="partitions_144",
         parquet="../20250911_235830/consumer/consumer-sts-0_consumer-result/processed/2025-09-11/batch_consumer-sts-*.parquet",
         blockmax="../20250911_235830/merge/merge-sts-0_merge-metrics/2025-09-11_13-02-52/block_maxima.csv",
     ),
     dict(
         label="partitions_288",
         parquet="../20250912_184917/consumer/consumer-sts-0_consumer-result/processed/2025-09-12/batch_consumer-sts-*.parquet",
         blockmax="../20250912_184917/merge/merge-sts-0_merge-metrics/2025-09-12_00-12-03/block_maxima.csv",
     )
]
# Tail thresholds & SLOs (tune if needed)
Lpc, Le2e, Lca = 0.25, 0.50, 0.05      # seconds
slo_pc = slo_e2e = slo_ca = 0.01       # ≤1% allowed over L
cap_max_s = 5.0                        # ignore obviously bogus >5s from synthetic runs

# ---------- Run analysis over all runs ----------
all_summaries = []
for r in RUNS:
    s = analyze_run(
        parquet_glob_or_dir=r["parquet"],
        blockmax_csv=r["blockmax"],
        label=r["label"],
        Lpc=Lpc, Le2e=Le2e, Lca=Lca,
        slo_pc=slo_pc, slo_e2e=slo_e2e, slo_ca=slo_ca,
        cap_max_s=cap_max_s, 
        warmup_sec=60,
    )
    all_summaries.append(s)

# Table per run
runs_table = pd.DataFrame([flatten(s) for s in all_summaries]).sort_values(
    ["passes_total","e2e_p99"], ascending=[False, True]
).reset_index(drop=True)

# Aggregate by NUM_PARTITIONS (handles multiple runs per setting)
def _agg_mean(series):  # safe mean that ignores NaNs
    return float(np.nanmean(series.values)) if len(series) else np.nan
by_partitions = (
    runs_table
    .groupby("num_partitions", dropna=False)
    .agg({
        "label": "count",
        "e2e_p99": _agg_mean,
        "pc_p99": _agg_mean,
        "ca_p99": _agg_mean,
        "e2e_frac_over_L": _agg_mean,
        "pc_frac_over_L": _agg_mean,
        "ca_frac_over_L": _agg_mean,
        "throughput_MBps": _agg_mean,
        "duplicate_rate": _agg_mean,
        "out_of_order_rate": _agg_mean,
        "n_msgs": "sum",
        "passes_total": "sum",
    })
    .rename(columns={"label": "n_runs"})
    .sort_values(["passes_total","e2e_p99"], ascending=[False, True])
)

best_run = choose_best_by_latency_and_slo(all_summaries)


print("Recommended (per-run comparison):", best_run["label"],
      f"| NUM_PARTITIONS={best_run['num_partitions']}, e2e p99={best_run['producer_application']['p99']:.4f}s, "
      f"passes={best_run['_passes']}/3, thr≈{best_run['health']['throughput_MBps']:.3f} MB/s")

print("\n=== Per-run table ===")
display(runs_table[
    ["label","num_partitions","max_messages","target_rate",
     "e2e_p99","pc_p99","ca_p99",
     "e2e_frac_over_L","pc_frac_over_L","ca_frac_over_L",
     "throughput_MBps","duplicate_rate","out_of_order_rate",
     "passes_total","parquet","metrics_dir"]
])

print("\n=== Grouped by NUM_PARTITIONS (averaged across runs) ===")
display(by_partitions)


Recommended (per-run comparison): partitions_288 | NUM_PARTITIONS=288, e2e p99=0.7154s, passes=1/3, thr≈13.634 MB/s

=== Per-run table ===


,label,num_partitions,max_messages,target_rate,e2e_p99,pc_p99,ca_p99,e2e_frac_over_L,pc_frac_over_L,ca_frac_over_L,throughput_MBps,duplicate_rate,out_of_order_rate,passes_total,parquet,metrics_dir
0,partitions_288,288,5000,250,0.715412,0.700397,0.0199,0.256236,0.717575,0.0,13.634392,0.0,0.0,1,../20250912_184917/consumer/consumer-sts-0_con...,../20250912_184917/merge/merge-sts-0_merge-met...
1,partitions_36,36,5000,250,0.748349,0.733342,0.0199,0.249564,0.711574,0.0,13.500539,0.0,0.0,1,../20250910_143637/consumer/consumer-sts-0_con...,../20250910_143637/merge/merge-sts-0_merge-met...
2,partitions_72,72,5000,250,0.755732,0.740710,0.0199,0.248867,0.710157,0.0,13.496595,0.0,0.0,1,../20250911_125217/consumer/consumer-sts-0_con...,../20250911_125217/merge/merge-sts-0_merge-met...
3,partitions_144,144,5000,250,0.800949,0.785936,0.0199,0.264378,0.722733,0.0,13.507883,0.0,0.0,1,../20250911_235830/consumer/consumer-sts-0_con...,../20250911_235830/merge/merge-sts-0_merge-met...



=== Grouped by NUM_PARTITIONS (averaged across runs) ===


,n_runs,e2e_p99,pc_p99,ca_p99,e2e_frac_over_L,pc_frac_over_L,ca_frac_over_L,throughput_MBps,duplicate_rate,out_of_order_rate,n_msgs,passes_total
num_partitions,,,,,,,,,,,,
288,1,0.715412,0.700397,0.0199,0.256236,0.717575,0.0,13.634392,0.0,0.0,29114072,1
36,1,0.748349,0.733342,0.0199,0.249564,0.711574,0.0,13.500539,0.0,0.0,21629238,1
72,1,0.755732,0.740710,0.0199,0.248867,0.710157,0.0,13.496595,0.0,0.0,22413575,1
144,1,0.800949,0.785936,0.0199,0.264378,0.722733,0.0,13.507883,0.0,0.0,16956186,1
